In [1]:

import json
import pennylane.numpy as np

def beam_splitter(r):
    """
    Returns the beam splitter matrix.

    Args:
        - r (float): The reflection coefficient of the beam splitter.
    Returns:
        - (np.array): 2 x 2 matrix that represents the beam
        splitter matrix.    
    """
    t = np.sqrt(1 - r ** 2)
    B = np.array([[r, t],
                  [t, -r]])
    return B

def mz_interferometer(r):
    """
    This function returns the probability that either A or C
    detect a photon, and the probability that D detects a photon.
    
    Args:
        - r (float): The reflection coefficient of the beam splitters.
    Returns: 
        - np.array(float): An array of shape (2,), where the first 
        element is the probability of detection at A or C,
        and the second element is the probability of detection at D.
    """
    B = beam_splitter(r)
    # Initial state |0>: amplitude vector [1,0]
    # After BS1: state is B @ [1,0] = column 0 of B
    a = B[0,0]  # amplitude for |0> (path 0, could be detected at A)
    b = B[1,0]  # amplitude for |1> (path 1, not detected at A)
    p_A = abs(a)**2
    # If not detected at A, photon is in |1>. After BS2, amplitudes are column 1 of B
    alpha_C = B[0,1]  # amplitude for |0> (C)
    alpha_D = B[1,1]  # amplitude for |1> (D)
    p_C = abs(b)**2 * abs(alpha_C)**2  # probability for C
    p_D = abs(b)**2 * abs(alpha_D)**2  # probability for D
    p_AC = p_A + p_C
    return np.array([p_AC, p_D])

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = mz_interferometer(ins).tolist()
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol=1e-4), "Not the correct probabilities"

def test_mz_interferometer():
    # Public test cases from problem
    inp = 0.5
    expected = [0.8125, 0.1875]
    output = mz_interferometer(inp)
    assert np.allclose(output, expected, rtol=1e-4), f"Failed for input {inp}: got {output}, expected {expected}"

    inp = 0.577350269
    expected = [0.777778, 0.222222]
    output = mz_interferometer(inp)
    assert np.allclose(output, expected, rtol=1e-4), f"Failed for input {inp}: got {output}, expected {expected}"

    # Edge cases: r = 0 (perfect transmission)
    inp = 0.0
    # At r=0, B=[[0,1],[1,0]] (swap gate); after first BS, all photon in |1>, through BS again: flips to |0> (C). So p_A=0, p_C=1, p_D=0.
    expected = [1.0, 0.0]
    output = mz_interferometer(inp)
    assert np.allclose(output, expected, rtol=1e-4), f"Failed for input {inp}: got {output}, expected {expected}"

    # Edge case: r=1 (perfect reflection), B=[[1,0],[0,-1]], state never leaves |0> path
    inp = 1.0
    expected = [1.0, 0.0]
    output = mz_interferometer(inp)
    assert np.allclose(output, expected, rtol=1e-4), f"Failed for input {inp}: got {output}, expected {expected}"

    print("All test cases passed!")

test_cases = [
    ('0.5', '[0.8125, 0.1875]'),
    ('0.577350269', '[0.777778, 0.222222]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
        except AssertionError as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
            print(e)
        else:
            print("Correct!")

# Uncomment to run additional internal tests:
# test_mz_interferometer()


Running test case 0 with input '0.5'...
Correct!
Running test case 1 with input '0.577350269'...
Correct!
